# LeetCode 598 — Range Addition II bằng Python

**Link đề:** https://leetcode.com/problems/range-addition-ii/
**Độ khó:** Easy · **Chủ đề:** Array, Math

```python
class Solution:
    def maxCount(self, m: int, n: int, ops: list[list[int]]) -> int:
        ...
```

## Mục tiêu học
1. Đọc hiểu thao tác trên **ma trận** (list 2 chiều) và tự mô phỏng được các phép cộng hình chữ nhật.
2. Biết bẫy kinh điển `[[0] * n] * m` (các hàng dùng chung một list) và cách tạo ma trận đúng.
3. Đi từ **mô phỏng** `O(k·m·n)` → **mảng hiệu 2 chiều (2D difference array)** `O(k + m·n)` → **giao các hình chữ nhật** `O(k)`.
4. Chứng minh vì sao đáp án là `min(a) * min(b)`, trace từng bước, phân tích Time/Space.
5. Làm 6 bài tập (có biến thể Range Addition 1D) với gợi ý 3 mức và đáp án cuối notebook.

## 1. Đề bài và ví dụ

Cho ma trận `M` kích thước `m × n`, ban đầu toàn `0`, và danh sách thao tác `ops`. Mỗi `ops[i] = [a, b]` nghĩa là: **cộng 1** vào mọi ô `M[x][y]` với `0 <= x < a` và `0 <= y < b` – tức là hình chữ nhật ở **góc trên-trái** có `a` hàng, `b` cột.

Sau khi làm hết thao tác, hãy **đếm số ô có giá trị lớn nhất**.

| m, n | ops | Output |
|---|---|---:|
| 3, 3 | `[[2,2],[3,3]]` | `4` |
| 3, 3 | `[[2,2],[3,3],[3,3],[3,3],[2,2],[3,3],[3,3],[3,3],[2,2],[3,3],[3,3],[3,3]]` | `4` |
| 3, 3 | `[]` | `9` |

Ví dụ 1 bằng hình:

```text
ban đầu     sau [2,2]    sau [3,3]
0 0 0       1 1 0        2 2 1
0 0 0  ->   1 1 0   ->   2 2 1
0 0 0       0 0 0        1 1 1
```
Giá trị lớn nhất là `2`, có **4** ô.

**Constraints:** `1 <= m, n <= 4·10^4`; `0 <= ops.length <= 10^4`; `1 <= a <= m`; `1 <= b <= n`.

Chú ý: `m·n` có thể tới `1.6·10^9` ô → **không thể** tạo ma trận thật. Ta cần suy luận.

## 2. Nền tảng: ma trận trong Python

### 2.1. List lồng list
Ma trận = list các hàng, mỗi hàng là một list. `M[x][y]` = hàng `x`, cột `y`.

Cách tạo đúng: **list comprehension** `[[0] * n for _ in range(m)]` – mỗi vòng tạo **một list hàng mới**.

In [1]:
m, n = 3, 4
M = [[0] * n for _ in range(m)]
M[1][2] = 7
for row in M:
    print(row)

[0, 0, 0, 0]
[0, 0, 7, 0]
[0, 0, 0, 0]


### 2.2. Bẫy `[[0] * n] * m`
`[row] * m` lặp lại **cùng một đối tượng list** `m` lần (copy tham chiếu, không copy nội dung). Sửa một hàng → mọi hàng đổi theo!

`id(x)` cho "địa chỉ" của đối tượng; hai hàng có cùng `id` nghĩa là cùng một list.

In [2]:
bad = [[0] * 4] * 3
bad[1][2] = 7
for row in bad:
    print(row)
print("id các hàng:", [id(r) for r in bad])
print("Hàng 0 và hàng 1 là cùng một list?", bad[0] is bad[1])

[0, 0, 7, 0]
[0, 0, 7, 0]
[0, 0, 7, 0]
id các hàng: [140278806073600, 140278806073600, 140278806073600]
Hàng 0 và hàng 1 là cùng một list? True


### 2.3. Vòng lặp hai tầng và `max`, `sum` trên ma trận
- Duyệt mọi ô: `for x in range(m): for y in range(n): ...`.
- Giá trị lớn nhất: `max(max(row) for row in M)`.
- Đếm số ô bằng `v`: `sum(row.count(v) for row in M)`.

In [3]:
M = [[2, 2, 1], [2, 2, 1], [1, 1, 1]]
mx = max(max(row) for row in M)
print("max =", mx)
print("số ô bằng max =", sum(row.count(mx) for row in M))

max = 2
số ô bằng max = 4


## 3. Cách 1 — Mô phỏng trực tiếp (baseline)

Làm đúng như đề: tạo ma trận, mỗi thao tác cộng 1 vào hình chữ nhật, cuối cùng đếm.

In [4]:
def max_count_simulate(m, n, ops):
    M = [[0] * n for _ in range(m)]
    for a, b in ops:                    # mở gói cặp [a, b]
        for x in range(a):
            for y in range(b):
                M[x][y] += 1
    mx = max(max(row) for row in M)
    return sum(row.count(mx) for row in M)


print(max_count_simulate(3, 3, [[2, 2], [3, 3]]))   # 4
print(max_count_simulate(3, 3, [[2, 2], [3, 3], [3, 3], [3, 3], [2, 2], [3, 3], [3, 3], [3, 3], [2, 2], [3, 3], [3, 3], [3, 3]]))  # 4
print(max_count_simulate(3, 3, []))                 # 9

4
4
9


### 3.1. TRACE mô phỏng
In ma trận sau mỗi thao tác để thấy vùng nào "được cộng nhiều nhất".

In [5]:
def trace_simulate(m, n, ops):
    M = [[0] * n for _ in range(m)]
    for step, (a, b) in enumerate(ops, 1):
        for x in range(a):
            for y in range(b):
                M[x][y] += 1
        print(f"Sau thao tác {step} = [{a},{b}]:")
        for row in M:
            print("   ", row)
    mx = max(max(row) for row in M)
    cnt = sum(row.count(mx) for row in M)
    print("max =", mx, "| số ô max =", cnt)
    return cnt

trace_simulate(3, 4, [[2, 3], [3, 2], [3, 4]])

Sau thao tác 1 = [2,3]:
    [1, 1, 1, 0]
    [1, 1, 1, 0]
    [0, 0, 0, 0]
Sau thao tác 2 = [3,2]:
    [2, 2, 1, 0]
    [2, 2, 1, 0]
    [1, 1, 0, 0]
Sau thao tác 3 = [3,4]:
    [3, 3, 2, 1]
    [3, 3, 2, 1]
    [2, 2, 1, 1]
max = 3 | số ô max = 4


4

**Nút thắt:** thời gian `O(k · m · n)` trong tệ nhất (`k` = số thao tác), bộ nhớ `O(m · n)`. Với `m = n = 4·10^4` thì riêng ma trận đã 1.6 tỉ ô – không thể. Ta cần bớt dần: trước hết bỏ phần "cộng từng ô của mỗi hình chữ nhật".

## 4. Cách 2 — Mảng hiệu 2 chiều (2D difference array)

### 4.1. Mảng hiệu 1 chiều trước
Muốn cộng `+1` cho đoạn `[l, r)` của mảng nhiều lần, thay vì cộng từng phần tử, ta ghi `diff[l] += 1`, `diff[r] -= 1`. Cuối cùng lấy **tổng tiền tố (prefix sum)** của `diff` là ra mảng thật.

In [6]:
size = 6
diff = [0] * (size + 1)
for l, r in [(0, 3), (2, 5)]:      # cộng 1 cho [0,3) và [2,5)
    diff[l] += 1
    diff[r] -= 1
print("diff  :", diff)

arr, running = [], 0
for i in range(size):
    running += diff[i]
    arr.append(running)
print("mảng  :", arr)            # [1, 1, 2, 1, 1, 0]

diff  : [1, 0, 1, -1, 0, -1, 0]
mảng  : [1, 1, 2, 1, 1, 0]


### 4.2. Áp dụng cho hình chữ nhật góc trên-trái
Hình chữ nhật `[0, a) × [0, b)` tương ứng 4 điểm ghi: `D[0][0] += 1`, `D[a][0] -= 1`, `D[0][b] -= 1`, `D[a][b] += 1`. Prefix sum 2 chiều:

`M[x][y] = D[x][y] + M[x-1][y] + M[x][y-1] − M[x-1][y-1]`.

Mỗi thao tác chỉ còn `O(1)`; dựng lại ma trận `O(m·n)`. Vẫn cần ma trận nhưng **không** phụ thuộc kích thước hình chữ nhật. Đây là kỹ thuật tổng quát khi các hình chữ nhật **bất kỳ** (không nhất thiết ở góc).

In [7]:
def max_count_diff2d(m, n, ops):
    D = [[0] * (n + 1) for _ in range(m + 1)]
    for a, b in ops:
        D[0][0] += 1
        D[a][0] -= 1
        D[0][b] -= 1
        D[a][b] += 1
    M = [[0] * n for _ in range(m)]
    for x in range(m):
        for y in range(n):
            up = M[x - 1][y] if x > 0 else 0
            left = M[x][y - 1] if y > 0 else 0
            diag = M[x - 1][y - 1] if x > 0 and y > 0 else 0
            M[x][y] = D[x][y] + up + left - diag
    mx = max(max(row) for row in M)
    return sum(row.count(mx) for row in M)


print(max_count_diff2d(3, 3, [[2, 2], [3, 3]]))
print(max_count_diff2d(3, 3, []))
print(max_count_diff2d(3, 4, [[2, 3], [3, 2], [3, 4]]))

4
9
4


## 5. Insight: ô `(0,0)` luôn được cộng ở MỌI thao tác

Mọi thao tác đều bắt đầu từ góc `(0, 0)` (vì `a, b >= 1`). Do đó:

1. Giá trị của ô `(0,0)` = `k` (số thao tác) – không ô nào lớn hơn được, vì mỗi thao tác chỉ cộng tối đa 1 vào mỗi ô. ⇒ **max = k**.
2. Một ô `(x, y)` đạt giá trị `k` ⇔ nó nằm trong **mọi** hình chữ nhật ⇔ `x < a_i` và `y < b_i` với **mọi** `i` ⇔ `x < min(a)` và `y < min(b)`.
3. ⇒ Số ô đạt max = `min(a) × min(b)` (giao của các hình chữ nhật góc trên-trái là một hình chữ nhật góc trên-trái).
4. Nếu `ops` rỗng: mọi ô bằng 0 = max → đáp án `m × n`. Có thể viết gọn bằng cách khởi tạo `min_a = m`, `min_b = n`.

Tính thử ví dụ 1: `min(2, 3) × min(2, 3) = 2 × 2 = 4` ✔.

In [8]:
ops = [[2, 3], [3, 2], [3, 4]]
min_a = min(a for a, b in ops)
min_b = min(b for a, b in ops)
print("min a =", min_a, "| min b =", min_b, "| đáp án =", min_a * min_b)

min a = 2 | min b = 2 | đáp án = 4


### 5.1. `min` với `default` và giá trị khởi tạo
`min(iterable)` báo lỗi `ValueError` nếu iterable rỗng. Hai cách tránh:
- `min(..., default=m)`;
- hoặc khởi tạo `min_a = m` rồi cập nhật trong vòng lặp (ops rỗng thì giữ nguyên `m`).

In [9]:
try:
    min([])
except ValueError as e:
    print("ValueError:", e)
print("min([], default=3) =", min([], default=3))

ValueError: min() iterable argument is empty
min([], default=3) = 3


## 6. Cách 3 — Giao các hình chữ nhật: giảng từng đoạn rồi ghép

- **Bước 1:** `min_a, min_b = m, n` (giao của "không có hình nào" là cả ma trận).
- **Bước 2:** duyệt `for a, b in ops:` cập nhật `min_a = min(min_a, a)`, `min_b = min(min_b, b)`.
- **Bước 3:** `return min_a * min_b`.

In [10]:
def max_count_intersection(m, n, ops):
    min_a, min_b = m, n
    for a, b in ops:
        min_a = min(min_a, a)
        min_b = min(min_b, b)
    return min_a * min_b


print(max_count_intersection(3, 3, [[2, 2], [3, 3]]))
print(max_count_intersection(3, 3, []))
print(max_count_intersection(40000, 40000, []))          # 1.6 tỉ ô – tính tức thì

4
9
1600000000


### 6.1. TRACE cách giao

Ví dụ `m = 3, n = 4, ops = [[2,3],[3,2],[3,4]]`:

| Bước | thao tác `[a,b]` | min_a | min_b | vùng giao hiện tại |
|---:|---|---:|---:|---|
| 0 | (khởi tạo) | 3 | 4 | cả ma trận 3×4 |
| 1 | `[2,3]` | 2 | 3 | 2×3 |
| 2 | `[3,2]` | 2 | 2 | 2×2 |
| 3 | `[3,4]` | 2 | 2 | 2×2 |

Đáp án `2 × 2 = 4`; trùng với trace mô phỏng ở mục 3.1 (4 ô có giá trị 3).

In [11]:
def trace_intersection(m, n, ops):
    min_a, min_b = m, n
    print(f"Bước 0: khởi tạo min_a={min_a}, min_b={min_b}")
    for step, (a, b) in enumerate(ops, 1):
        min_a, min_b = min(min_a, a), min(min_b, b)
        print(f"Bước {step}: op=[{a},{b}] -> min_a={min_a}, min_b={min_b}, giao = {min_a}x{min_b}")
    print("Đáp án =", min_a * min_b)
    return min_a * min_b

trace_intersection(3, 4, [[2, 3], [3, 2], [3, 4]])

Bước 0: khởi tạo min_a=3, min_b=4
Bước 1: op=[2,3] -> min_a=2, min_b=3, giao = 2x3
Bước 2: op=[3,2] -> min_a=2, min_b=2, giao = 2x2
Bước 3: op=[3,4] -> min_a=2, min_b=2, giao = 2x2
Đáp án = 4


4

## 7. Code nộp LeetCode

In [12]:
class Solution:
    def maxCount(self, m: int, n: int, ops: list[list[int]]) -> int:
        min_a, min_b = m, n
        for a, b in ops:
            if a < min_a:
                min_a = a
            if b < min_b:
                min_b = b
        return min_a * min_b


sol = Solution()
assert sol.maxCount(3, 3, [[2, 2], [3, 3]]) == 4
assert sol.maxCount(3, 3, [[2, 2], [3, 3], [3, 3], [3, 3], [2, 2], [3, 3], [3, 3], [3, 3], [2, 2], [3, 3], [3, 3], [3, 3]]) == 4
assert sol.maxCount(3, 3, []) == 9
print("Solution PASS ví dụ đề")

Solution PASS ví dụ đề


Phiên bản một dòng (dùng `default`):

```python
class Solution:
    def maxCount(self, m: int, n: int, ops: list[list[int]]) -> int:
        return min((a for a, _ in ops), default=m) * min((b for _, b in ops), default=n)
```

In [13]:
class SolutionOneLine:
    def maxCount(self, m: int, n: int, ops: list[list[int]]) -> int:
        return min((a for a, _ in ops), default=m) * min((b for _, b in ops), default=n)

assert SolutionOneLine().maxCount(3, 3, [[2, 2], [3, 3]]) == 4
assert SolutionOneLine().maxCount(3, 3, []) == 9
print("SolutionOneLine PASS")

SolutionOneLine PASS


## 8. Vì sao đúng? (chứng minh ngắn)

Gọi `k = len(ops)`. Với ô `(x, y)`, giá trị cuối `M[x][y]` = số thao tác `i` thỏa `x < a_i` và `y < b_i`.

1. `M[x][y] <= k` với mọi ô, và `M[0][0] = k` (vì `a_i, b_i >= 1`). ⇒ giá trị lớn nhất **đúng bằng** `k`.
2. `M[x][y] = k` ⇔ mọi thao tác đều chứa `(x, y)` ⇔ `x < min_i a_i` và `y < min_i b_i`.
3. Số cặp `(x, y)` như vậy là `min_a × min_b` (vì `min_a <= m`, `min_b <= n`, các ô này đều nằm trong ma trận).
4. Trường hợp `k = 0`: max = 0 và mọi `m·n` ô đều bằng 0; công thức với khởi tạo `min_a = m, min_b = n` cho đúng `m·n`.

**Invariant của vòng lặp:** sau khi xử lý `i` thao tác đầu, `[0, min_a) × [0, min_b)` chính là giao của `i` hình chữ nhật đầu (và cả ma trận).

## 9. Time / Space Complexity và so sánh

`k` = số thao tác.

| Cách | Time | Space | Dùng khi |
|---|---|---|---|
| 1. Mô phỏng | `O(k·m·n)` tệ nhất | `O(m·n)` | Chỉ để học / kiểm tra nhỏ |
| 2. Mảng hiệu 2D | `O(k + m·n)` | `O(m·n)` | Hình chữ nhật **bất kỳ**, ma trận vừa phải |
| 3. Giao hình chữ nhật | `O(k)` | `O(1)` | Hình chữ nhật đều chung góc `(0,0)` – bài này |

Cách 3 không phụ thuộc `m, n` – vì vậy tính được cả khi ma trận 1.6 tỉ ô.

## 10. Bộ test: mẫu đề, biên, đối chiếu ngẫu nhiên

In [14]:
import random

methods = {
    "simulate": max_count_simulate,
    "diff2d": max_count_diff2d,
    "intersection": max_count_intersection,
    "Solution": Solution().maxCount,
}
cases = [
    ("đề 1", 3, 3, [[2, 2], [3, 3]], 4),
    ("đề 2", 3, 3, [[2, 2], [3, 3], [3, 3], [3, 3], [2, 2], [3, 3], [3, 3], [3, 3], [2, 2], [3, 3], [3, 3], [3, 3]], 4),
    ("đề 3 ops rỗng", 3, 3, [], 9),
    ("ma trận 1x1", 1, 1, [[1, 1]], 1),
    ("một op phủ hết", 2, 5, [[2, 5]], 10),
    ("op nhỏ nhất 1x1", 4, 4, [[4, 4], [1, 1]], 1),
    ("min a, min b ở 2 op khác nhau", 5, 5, [[1, 5], [5, 2]], 2),
]
for name, m, n, ops, expected in cases:
    for mname, f in methods.items():
        got = f(m, n, ops)
        assert got == expected, (name, mname, got)
    print(f"{name:<30} PASS -> {expected}")

rng = random.Random(598)
for _ in range(300):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    ops = [[rng.randint(1, m), rng.randint(1, n)] for _ in range(rng.randint(0, 5))]
    expected = max_count_simulate(m, n, ops)
    for mname, f in methods.items():
        assert f(m, n, ops) == expected, (m, n, ops, mname)
print("300 bộ ngẫu nhiên: 4 cách khớp nhau — PASS")

đề 1                           PASS -> 4
đề 2                           PASS -> 4
đề 3 ops rỗng                  PASS -> 9
ma trận 1x1                    PASS -> 1
một op phủ hết                 PASS -> 10
op nhỏ nhất 1x1                PASS -> 1
min a, min b ở 2 op khác nhau  PASS -> 2
300 bộ ngẫu nhiên: 4 cách khớp nhau — PASS


## 11. Lỗi người mới hay gặp

### Lỗi 1: Tạo ma trận bằng `[[0] * n] * m`
Các hàng dùng chung → mô phỏng sai hoàn toàn.

In [15]:
def simulate_wrong_matrix(m, n, ops):
    M = [[0] * n] * m                   # SAI
    for a, b in ops:
        for x in range(a):
            for y in range(b):
                M[x][y] += 1
    mx = max(max(row) for row in M)
    return sum(row.count(mx) for row in M)

print("Sai :", simulate_wrong_matrix(3, 3, [[2, 2], [3, 3]]), "| Đúng:", max_count_simulate(3, 3, [[2, 2], [3, 3]]))

Sai : 6 | Đúng: 4


### Lỗi 2: Quên trường hợp `ops` rỗng
`min(a for a, b in ops)` → `ValueError` khi `ops = []`. Đáp án đúng là `m * n`.

### Lỗi 3: Khởi tạo `min_a = 0`
Khi đó `min(0, a) = 0` mãi → đáp án luôn 0. Phải khởi tạo bằng `m`, `n` (hoặc vô cực `float('inf')` rồi xử lý ops rỗng riêng).

### Lỗi 4: Nghĩ rằng phải tìm "ô bị cộng nhiều lần nhất" bằng cách đếm thao tác lớn
Hình chữ nhật **lớn** không làm tăng max; chính hình chữ nhật **nhỏ nhất theo từng chiều** quyết định vùng max. Và lưu ý min a, min b có thể đến từ **hai thao tác khác nhau** (test `[[1,5],[5,2]]` → 2).

In [16]:
def wrong_init_zero(m, n, ops):
    min_a = min_b = 0                   # SAI
    for a, b in ops:
        min_a, min_b = min(min_a, a), min(min_b, b)
    return min_a * min_b

def wrong_pick_smallest_area(m, n, ops):
    if not ops:
        return m * n
    a, b = min(ops, key=lambda op: op[0] * op[1])   # SAI: chọn 1 op diện tích nhỏ nhất
    return a * b

print("Khởi tạo 0       :", wrong_init_zero(3, 3, [[2, 2], [3, 3]]), "(đúng 4)")
print("Chọn op nhỏ nhất :", wrong_pick_smallest_area(5, 5, [[1, 5], [5, 2]]), "(đúng 2)")

Khởi tạo 0       : 0 (đúng 4)
Chọn op nhỏ nhất : 5 (đúng 2)


## 12. Mẹo nhận dạng pattern

- Nhiều thao tác cùng **neo ở một góc / một đầu** (prefix) → kết quả phụ thuộc vào thao tác **nhỏ nhất** (giao) hoặc **lớn nhất** (hợp).
- Đề cho kích thước khổng lồ (`10^9` ô) nhưng số thao tác nhỏ → đừng mô phỏng, hãy tìm **công thức** theo thao tác.
- Cộng trên đoạn/hình chữ nhật **bất kỳ** nhiều lần → **mảng hiệu (difference array)** + prefix sum.
- `key=lambda ...` trong `min/max/sorted`: chọn theo tiêu chí tự định nghĩa. `lambda op: op[0] * op[1]` là hàm vô danh nhận `op` trả diện tích.

In [17]:
ops = [[3, 1], [2, 2], [1, 4]]
print("op có a nhỏ nhất:", min(ops, key=lambda op: op[0]))
print("op diện tích nhỏ nhất:", min(ops, key=lambda op: op[0] * op[1]))

op có a nhỏ nhất: [1, 4]
op diện tích nhỏ nhất: [3, 1]


## 13. Biến thể

**Biến thể A — Range Addition (LeetCode 370, 1 chiều, có giá trị cộng):** cho độ dài `length` và các update `[l, r, inc]` (đoạn đóng `[l, r]`), trả về mảng cuối. → mảng hiệu 1D.

**Biến thể B — Hình chữ nhật bất kỳ** `[r1, c1, r2, c2]` (đóng): đếm số ô đạt max → mảng hiệu 2D (mục 4).

**Biến thể C — Giá trị lớn nhất là bao nhiêu?** Với bài gốc: chính là `len(ops)`.

**Biến thể D — Hình chữ nhật neo ở góc dưới-phải**: lập luận đối xứng, vẫn là `min × min`.

## 14. Bài tập (TODO – không gọi khi chưa làm)

**Bài 1 — Đoán output.** Đoạn sau in gì?
```python
g = [[0] * 2] * 2
g[0][0] = 5
print(g)
```

**Bài 2 — Tính tay.** `m = 4, n = 5, ops = [[3,4],[4,2],[2,5]]`: giá trị max là bao nhiêu và có bao nhiêu ô?

**Bài 3 — Viết lại** `maxCount` bằng cách giao hình chữ nhật.

**Bài 4 — Biến thể C:** trả về `(giá trị max, số ô max)`.

**Bài 5 — Range Addition 1D (LC 370):** `range_addition(length, updates)` với `updates[i] = [l, r, inc]`.

**Bài 6 — Mini challenge:** hình chữ nhật bất kỳ `[r1, c1, r2, c2]` (đóng hai đầu) trên ma trận `m × n`; đếm số ô đạt giá trị lớn nhất bằng mảng hiệu 2D.

In [18]:
answer_ex1 = None       # TODO: ghi list bạn đoán
answer_ex2 = None       # TODO: (max, số ô)


def max_count_exercise(m, n, ops):
    # TODO: khởi tạo min_a, min_b = m, n; cập nhật; trả về tích
    pass


def max_value_and_count(m, n, ops):
    # TODO: trả về tuple (max, count)
    pass

# Kiểm tra sau khi làm:
# assert max_count_exercise(3, 3, [[2, 2], [3, 3]]) == 4
# assert max_value_and_count(3, 3, [[2, 2], [3, 3]]) == (2, 4)

In [19]:
def range_addition(length, updates):
    # TODO: diff = [0] * (length + 1); diff[l] += inc; diff[r + 1] -= inc; rồi prefix sum
    pass


def count_max_any_rectangles(m, n, rects):
    # TODO: mảng hiệu 2D với 4 điểm ghi cho mỗi hình [r1, c1, r2, c2]
    pass

# Kiểm tra sau khi làm:
# assert range_addition(5, [[1, 3, 2], [2, 4, 3], [0, 2, -2]]) == [-2, 0, 3, 5, 3]
# assert count_max_any_rectangles(3, 3, [[0, 0, 1, 1], [1, 1, 2, 2]]) == 1

## 15. Gợi ý 3 mức

**Bài 1:** (1) Nhớ mục 2.2. (2) Hai hàng là cùng một list. (3) Cả hai hàng đều bắt đầu bằng 5.

**Bài 2:** (1) Max = số thao tác. (2) Lấy min theo hàng và min theo cột riêng. (3) `min(3,4,2) × min(4,2,5)`.

**Bài 3:** (1) Giao hình chữ nhật góc. (2) `for a, b in ops`. (3) Xem mục 6.

**Bài 4:** (1) Max chính là `len(ops)` – kể cả khi rỗng (=0). (2) Số ô dùng lại Bài 3. (3) `return (len(ops), max_count_exercise(...))`.

**Bài 5:** (1) Mảng hiệu: đầu đoạn cộng, sau cuối đoạn trừ. (2) Đoạn **đóng** `[l, r]` nên trừ ở `r + 1`, mảng dài `length + 1`. (3) Vòng prefix: `running += diff[i]; res.append(running)`.

**Bài 6:** (1) Như mục 4.2 nhưng góc không phải `(0,0)`. (2) Bốn điểm: `D[r1][c1] += 1`, `D[r2+1][c1] -= 1`, `D[r1][c2+1] -= 1`, `D[r2+1][c2+1] += 1`. (3) Prefix sum 2D rồi đếm số ô bằng max.

## 16. Đáp án tham khảo (chạy kiểm tra)

In [20]:
g = [[0] * 2] * 2
g[0][0] = 5
ans_ex1 = g                               # [[5, 0], [5, 0]]
ans_ex2 = (3, max_count_intersection(4, 5, [[3, 4], [4, 2], [2, 5]]))   # (3, 2*2=4)


def ans_max_count(m, n, ops):
    min_a, min_b = m, n
    for a, b in ops:
        min_a, min_b = min(min_a, a), min(min_b, b)
    return min_a * min_b


def ans_max_value_and_count(m, n, ops):
    return (len(ops), ans_max_count(m, n, ops))


def ans_range_addition(length, updates):
    diff = [0] * (length + 1)
    for l, r, inc in updates:
        diff[l] += inc
        diff[r + 1] -= inc
    res, running = [], 0
    for i in range(length):
        running += diff[i]
        res.append(running)
    return res


def ans_count_max_any_rectangles(m, n, rects):
    D = [[0] * (n + 1) for _ in range(m + 1)]
    for r1, c1, r2, c2 in rects:
        D[r1][c1] += 1
        D[r2 + 1][c1] -= 1
        D[r1][c2 + 1] -= 1
        D[r2 + 1][c2 + 1] += 1
    M = [[0] * n for _ in range(m)]
    for x in range(m):
        for y in range(n):
            M[x][y] = (D[x][y]
                       + (M[x - 1][y] if x else 0)
                       + (M[x][y - 1] if y else 0)
                       - (M[x - 1][y - 1] if x and y else 0))
    mx = max(max(row) for row in M)
    return sum(row.count(mx) for row in M)


print("Bài 1:", ans_ex1, "| Bài 2:", ans_ex2)
assert ans_ex1 == [[5, 0], [5, 0]] and ans_ex2 == (3, 4)
assert ans_max_count(3, 3, [[2, 2], [3, 3]]) == 4 and ans_max_count(3, 3, []) == 9
assert ans_max_value_and_count(3, 3, [[2, 2], [3, 3]]) == (2, 4)
assert ans_max_value_and_count(2, 2, []) == (0, 4)
assert ans_range_addition(5, [[1, 3, 2], [2, 4, 3], [0, 2, -2]]) == [-2, 0, 3, 5, 3]
assert ans_count_max_any_rectangles(3, 3, [[0, 0, 1, 1], [1, 1, 2, 2]]) == 1
# Hình góc trên-trái là trường hợp riêng của bài 6:
assert ans_count_max_any_rectangles(3, 3, [[0, 0, 1, 1], [0, 0, 2, 2]]) == 4
print("Đáp án: tất cả test PASS")

Bài 1: [[5, 0], [5, 0]] | Bài 2: (3, 4)
Đáp án: tất cả test PASS


## 17. Template nhớ nhanh và checklist submit

```python
class Solution:
    def maxCount(self, m: int, n: int, ops: list[list[int]]) -> int:
        min_a, min_b = m, n          # ops rỗng -> cả ma trận
        for a, b in ops:
            min_a = min(min_a, a)
            min_b = min(min_b, b)
        return min_a * min_b         # giao của mọi hình chữ nhật góc (0,0)
```

Câu nhớ: **"mọi thao tác đều chạm (0,0) → max = số thao tác → vùng max = giao = min × min"**.

### Checklist
- [ ] `ops` rỗng trả `m * n`?
- [ ] Khởi tạo `min_a = m`, `min_b = n` (không phải 0)?
- [ ] Lấy min **riêng** từng chiều (không chọn một thao tác diện tích nhỏ nhất)?
- [ ] Không tạo ma trận thật (có thể tới 1.6 tỉ ô)?
- [ ] Nếu cần ma trận ở bài khác: tạo bằng `[[0] * n for _ in range(m)]`?